# MOMENT fcst: Temporal Feature Attack (DFT + trend, gray-box, kiểu Koren) (TV4)

Chạy **sau** `moment_fcst_colab.ipynb` (cần `ckpt/foundation_fcst/full_<level>_s42.done.json` và `full_<level>_s42_preds_ep10.pt`). Notebook này **không cần GPU** và không cần weight của model.

Ý tưởng: residual `r = target - dự đoán` của bản ghi member thường "sạch" hơn (ít cấu trúc còn sót) so với non-member. Thay vì chỉ nhìn độ lớn (Loss), attacker nhìn **hình dạng** residual:
1. **DFT**: biên độ phổ của residual theo từng lead, gộp thành 8 dải tần (log). Bắt phần dư có chu kỳ, ví dụ model bỏ sót nhịp QRS/sóng T.
2. **Trend**: độ dốc của residual và của |residual| theo horizon (sai số có tăng dần về cuối không).
3. **Log MSE** theo từng lead.

Gray-box: attacker có một ít nhãn member/non-member (cross-fitting 5-fold): mỗi bản ghi được chấm điểm bởi bộ phân loại chưa từng thấy nó. Có kèm 2 ablation: cùng đặc trưng nhưng classifier tuyến tính, và chỉ dùng Loss qua cùng pipeline, để thấy DFT/trend đóng góp thêm bao nhiêu so với Loss.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # must be set before torch touches CUDA
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Final_Project'
MOMENT_SRC = f'{DRIVE_ROOT}/moment'          # folder containing setup.py
DATASET_DIR = f'{DRIVE_ROOT}/dataset'        # shared split script + splits/
PTBXL_ZIP = f'{DATASET_DIR}/archive (3).zip'
RESULTS_DIR = f'{DRIVE_ROOT}/results/foundation/fcst'

In [ ]:
# Copy source to local disk (faster than installing from Drive), then install
!rm -rf /content/moment && cp -r {MOMENT_SRC} /content/moment
!pip install -q /content/moment wfdb scikit-learn pandas tqdm

## 1. Cấu hình

In [ ]:
SEED = 42
LEVEL = 'record'            # record-level split, must match the fcst run
MODEL_NAME = 'AutonLab/MOMENT-1-large'
MODE = 'full'
LOOKBACK, HORIZON = 512, 96
TARGET_EPOCH = 10           # which preds_ep{n}.pt from moment_fcst_colab to attack

N_VAL = 500
N_BOOT = 1000
N_FOLDS = 5                 # cross-fitting folds for the attack classifier
N_BANDS = 8                 # DFT magnitude is pooled into this many frequency bands per lead
CKPT_DIR = f'{DRIVE_ROOT}/ckpt/foundation_fcst'
TAG = f'{MODE}_{LEVEL}_s{SEED}'

PTBXL_ROOT = '/content/ptbxl_raw'
SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
CACHE_DIR = f'/content/cache/fcst_{LEVEL}_s{SEED}'

## 2. Dữ liệu PTB-XL
Giải nén `archive (3).zip` từ Drive vào disk local của Colab (đọc nhanh hơn Drive).

In [ ]:
import os, glob
if not glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    assert os.path.exists(PTBXL_ZIP), f'missing {PTBXL_ZIP}; {DATASET_DIR} has {os.listdir(DATASET_DIR)}'
    !unzip -q -o "{PTBXL_ZIP}" -d {PTBXL_ROOT}
found = glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True)
assert found, f'no ptbxl_database.csv after unzip; {PTBXL_ROOT} has {os.listdir(PTBXL_ROOT)}'
PTBXL_DIR = os.path.dirname(found[0])
print(PTBXL_DIR, os.listdir(PTBXL_DIR))

In [ ]:
import json, random, subprocess, sys
import numpy as np, torch, pandas as pd
from momentfm.data.ptbxl_classification_dataset import PTBXL_dataset

def control_randomness(seed=SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
control_randomness()

# shared split from dataset/make_split.py (generated here if missing)
if not os.path.exists(SPLIT_PATH):
    subprocess.run([sys.executable, f'{DATASET_DIR}/make_split.py', '--ptbxl-dir', PTBXL_DIR,
                    '--level', LEVEL, '--seed', str(SEED), '--n-val', str(N_VAL)], check=True)
split = json.load(open(SPLIT_PATH))
meta = json.load(open(SPLIT_PATH.replace('.json', '.meta.json')))
print({k: len(v) for k, v in split.items()}, 'sha256', meta['sha256'][:12])
if len(split['test']) < 10000:
    print('WARNING: fewer than 10k non-members; TPR@0.1%FPR rests on', int(0.001 * len(split['test'])), 'samples')

class Config:
    base_path = PTBXL_DIR
    cache_dir = CACHE_DIR
    path_to_pids = SPLIT_PATH
    load_cache = True
    fs = 100
    output_type = 'single'    # lowercase: modify_df compares against 'single'
    seq_len = LOOKBACK + HORIZON   # record is resampled to this length, then split into input / target
os.makedirs(CACHE_DIR, exist_ok=True)

member_ds = PTBXL_dataset(Config, phase='train')
val_ds    = PTBXL_dataset(Config, phase='val')
nonmem_ds = PTBXL_dataset(Config, phase='test')
print(len(member_ds), len(val_ds), len(nonmem_ds), member_ds.timeseries.shape)

## 2. Residual + đặc trưng temporal

In [ ]:
import torch
from scipy.stats import rankdata

done = json.load(open(f'{CKPT_DIR}/{TAG}.done.json'))          # fails loudly if fcst training never finished
pr = torch.load(f'{CKPT_DIR}/{TAG}_preds_ep{TARGET_EPOCH}.pt')
tgt_m = member_ds.timeseries[..., LOOKBACK:].astype(np.float32)
tgt_n = nonmem_ds.timeseries[..., LOOKBACK:].astype(np.float32)
res_m = tgt_m - pr['pm'].numpy()                               # residuals [N, 12, HORIZON]
res_n = tgt_n - pr['pn'].numpy()
assert res_m.shape[1:] == (12, HORIZON) and len(res_m) == len(member_ds) and len(res_n) == len(nonmem_ds)

def slope(v):
    # least-squares slope over the horizon axis, per row of v [..., H]
    t = np.arange(v.shape[-1], dtype=np.float32); t -= t.mean()
    return (v * t).sum(-1) / (t ** 2).sum()

def temporal_features(res):
    n, c, h = res.shape
    mag = np.abs(np.fft.rfft(res, axis=-1))[..., 1:]           # drop the DC bin (it is the mean = bias)
    bands = np.array_split(np.arange(mag.shape[-1]), N_BANDS)
    dft = np.stack([np.log1p(mag[..., b].mean(-1)) for b in bands], -1).reshape(n, -1)   # DFT: [n, 12*N_BANDS]
    trend = slope(res)                                         # trend of the signed residual: [n, 12]
    drift = slope(np.abs(res))                                 # does the error grow along the horizon: [n, 12]
    lmse = np.log((res ** 2).mean(-1) + 1e-8)                  # per-lead log MSE: [n, 12]
    return np.concatenate([dft, trend, drift, lmse], 1)

Xm, Xn = temporal_features(res_m), temporal_features(res_n)
X = np.r_[Xm, Xn]; y = np.r_[np.ones(len(Xm)), np.zeros(len(Xn))]
loss_all = -np.r_[(res_m ** 2).mean((1, 2)), (res_n ** 2).mean((1, 2))]
print('features', X.shape, 'members', len(Xm), 'non-members', len(Xn))

## 3. Tấn công (cross-fitting) + ablation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc as sk_auc

def tpr_at(fpr, tpr, t):
    m = fpr <= t
    return float(tpr[m].max()) if m.any() else 0.0

def mia_metrics(sm, sn):
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    return tpr_at(fpr, tpr, 1e-3), tpr_at(fpr, tpr, 1e-2), float(sk_auc(fpr, tpr))

def bootstrap_ci(sm, sn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    res = np.array([mia_metrics(sm[rng.integers(0, len(sm), len(sm))],
                                sn[rng.integers(0, len(sn), len(sn))])[:2] for _ in range(n)])
    lo, hi = np.percentile(res, [2.5, 97.5], axis=0)
    return [float(lo[0]), float(hi[0])], [float(lo[1]), float(hi[1])]

def cross_fit(make_clf, X, y):
    # every sample is scored by a classifier that never saw it, so scores of members and non-members are comparable
    score = np.zeros(len(y))
    for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y):
        clf = make_clf().fit(X[tr], y[tr])
        score[te] = clf.predict_proba(X[te])[:, 1]
    return score

makers = {
    # main attack, fixed in advance: gradient boosting on DFT + trend + drift + per-lead log MSE
    'temporal':      (lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED), X),
    # ablation 1: same features, linear classifier
    'temporal_lr':   (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X),
    # ablation 2: same pipeline with only the scalar loss, shows what DFT/trend add over Loss
    'loss_only_clf': (lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED), loss_all[:, None]),
}
S, M = {}, {}
nm = len(Xm)
for name, (mk, feats) in makers.items():
    S[name] = cross_fit(mk, feats, y)
    M[name] = mia_metrics(S[name][:nm], S[name][nm:])
M['loss (raw -MSE)'] = mia_metrics(loss_all[:nm], loss_all[nm:])
print(pd.DataFrame(M, index=['TPR@0.1%', 'TPR@1%', 'AUC']).T.round(4))
if len(Xn) < 10000:
    print('WARNING: fewer than 10k non-members; TPR@0.1%FPR rests on', int(0.001 * len(Xn)), 'samples')

## 4. Ghi kết quả

In [ ]:
sm, sn = S['temporal'][:nm], S['temporal'][nm:]
c01, c1 = bootstrap_ci(sm, sn)
t01, t1, au = M['temporal']
hist = done['history'][-1] if done.get('history') else {}
out = {
    'model': 'foundation', 'task': 'fcst', 'attack': 'temporal',
    'owner_tv': 4, 'level': LEVEL, 'dataset': 'ptbxl', 'seed': SEED,
    'n_member': int(len(sm)), 'n_nonmember': int(len(sn)),
    'scores_member': np.round(sm, 6).tolist(), 'scores_nonmember': np.round(sn, 6).tolist(),
    'tpr_at_0.1fpr': t01, 'tpr_at_1fpr': t1, 'auc': au,
    'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
    'utility': {k: hist[k] for k in ('train_mse', 'test_mse', 'gap') if k in hist},
    'n_shadow': None,
    'extra': {'mode': MODE, 'eval_epoch': TARGET_EPOCH, 'lookback': LOOKBACK, 'horizon': HORIZON, 'backbone': MODEL_NAME,
              'classifier': 'HistGradientBoosting', 'n_folds': N_FOLDS, 'n_bands': N_BANDS,
              'features': 'log DFT band magnitude + residual trend + residual drift + per-lead log MSE (per lead)',
              'ablation': {k: {'tpr_at_0.1fpr': v[0], 'tpr_at_1fpr': v[1], 'auc': v[2]} for k, v in M.items()}},
}
os.makedirs(RESULTS_DIR, exist_ok=True)
fn = f'{RESULTS_DIR}/temporal_{MODE}.json'
json.dump(out, open(fn, 'w'))
print('saved', fn, '| TPR@0.1%', round(t01, 4), c01, '| TPR@1%', round(t1, 4), c1, '| AUC', round(au, 4))

## 5. ROC log–log

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 5))
for name, s in {'temporal (DFT+trend)': S['temporal'], 'temporal LR': S['temporal_lr'], 'loss only + clf': S['loss_only_clf'], 'raw loss': loss_all}.items():
    fpr, tpr, _ = roc_curve(y, s)
    ax.plot(fpr, tpr, label=name)
ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8); ax.set_title(f'MOMENT PTB-XL fcst, {LEVEL}-level, Temporal attack')
plt.show()